In [62]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from scipy.spatial.distance import cosine
from sklearn.preprocessing import StandardScaler, FunctionTransformer, QuantileTransformer

# Loading Data & Similarity Matrix

In [63]:
df = pd.read_csv(r"archive\CombinedClean.csv")

In [64]:
df = df.drop(["Recalled Word"], axis=1) 

In [65]:
word_sim_df = pd.read_csv("word_similarity_matrix2.csv")
word_sim_df.set_index("Unnamed: 0", inplace=True)

word_sim_df.index.name = None
for diag_idx in range(word_sim_df.shape[0]):
    word_sim_df.iloc[diag_idx, diag_idx] = 1

word_sim_matrix = word_sim_df.to_numpy()
word_vec_df = pd.read_csv("word_vec_fasttext2.csv")
word_vec_matrix = word_vec_df.T.to_numpy()

In [66]:
vector_mapping = {word: word_vec_df[word].values for word in word_vec_df.columns}
vector_mapping['MEKTUP']

array([-1.15079780e-01,  2.25585980e-02,  1.10532760e-01,  3.98818370e-02,
        1.72224470e-02, -7.89943700e-03,  4.55937830e-02,  8.63005150e-02,
        6.15905700e-02,  6.21303600e-02, -4.69180570e-02, -1.11103650e-03,
       -6.40903340e-02, -3.81387560e-03, -2.65922230e-02,  1.22322656e-01,
        3.47482300e-02, -4.45182840e-02,  1.33062770e-01, -2.70089800e-02,
       -2.11032460e-03,  1.44670920e-01, -5.82016900e-02,  6.59341800e-02,
        6.60292000e-02, -4.75344700e-02, -7.23616850e-02, -3.38157860e-02,
       -6.75574000e-05,  6.26461360e-02,  1.32613620e-02, -8.46384700e-02,
       -1.11362080e-01, -7.63710960e-02,  1.11168005e-01,  1.07916600e-02,
       -7.14420100e-02,  3.88509630e-02, -1.16184585e-01,  3.57477500e-02,
        2.65713730e-02,  1.98520800e-02,  6.67357400e-02,  6.56064700e-02,
        2.95802240e-02,  3.23552340e-03, -5.18452300e-02, -6.02071770e-02,
       -7.75517750e-02,  4.67715260e-02, -2.31263230e-02,  2.51263120e-02,
       -6.69591300e-02,  

# Data Cleaning & Features

In [67]:
df.columns

Index(['ListID', 'Presented Word', 'Present Position', 'Recall Position',
       'Reaction Time', 'Hit', 'Participant ID', 'Participant Group'],
      dtype='object')

For now, drop Reaction Time and Recall Position too. Maybe can run a regression task.

In [68]:
df = df.drop(["Reaction Time", "Recall Position"], axis=1)

In [69]:
df = df.drop(df[(df["ListID"] == 0)].index, axis=0) #get rid of the tutorial set
df["Hit"] = df["Hit"].replace(2, 0) #replace wrong hits with 0

In [70]:
df = df.sort_values(by=["Participant Group","Participant ID", "ListID","Present Position"]) #sort
df = df.reset_index()

In [71]:
df["Participant ID"] = df["Participant ID"].astype(str) + \
    df["Participant Group"].map({"eng":"_ENG", "soc":"_SOC"})

In [72]:
"""##Distance to Most Similar Item
#compare within lists
#use similarity matrix constructed 
last_list_id = df["ListID"][0]
last_participant_id = df["Participant ID"][0]
last_participant_group = df["Participant Group"][0]

for row in range(df.shape[0]): #iterating through rows
    # MOST SIMILAR VAL
    # MOST SIMILAR PRES DISTANCE
    while df["Participant ID"][row] == last_participant_id and df[""]"""

'##Distance to Most Similar Item\n#compare within lists\n#use similarity matrix constructed \nlast_list_id = df["ListID"][0]\nlast_participant_id = df["Participant ID"][0]\nlast_participant_group = df["Participant Group"][0]\n\nfor row in range(df.shape[0]): #iterating through rows\n    # MOST SIMILAR VAL\n    # MOST SIMILAR PRES DISTANCE\n    while df["Participant ID"][row] == last_participant_id and df[""]'

## Feature Extracting

### Position Zone

In [73]:
df["POSITION_ZONE"] = pd.cut(
    df["Present Position"],
    bins = [0,5,11,18],
    labels=[1,2,3]
).astype(int) #divide positions to zones

### Extract Similarity

In [74]:
def find_most_similiar(group, sim_df): 
    length = len(group)-1
    similarities = []
    distances = []
    pos = []
    words = group["Presented Word"].values #get rid of pandas indexing
    
    i = 0
    while i <= length: 
        most_sim_index = i 
        current_word = words[i]
        most_sim_val = -2

        j = 0
        while j < i: 
            comparison_word = words[j]
            try: 
                similarity = sim_df.loc[current_word, comparison_word]
                if pd.notna(similarity) and similarity > most_sim_val:
                    most_sim_val, most_sim_index = similarity, j
            except KeyError: 
                print (f"{current_word} or {comparison_word} is not found on object")
                pass
            j += 1

        if most_sim_val < -1: 
            similarities.append(np.nan)
            distances.append(np.nan)
            pos.append(np.nan)
        else:
            similarities.append(most_sim_val)
            distances.append(i-most_sim_index)
            pos.append(most_sim_index)
        i += 1
    return pd.DataFrame(data = {"MOST_SIM_VAL": similarities,
                             "MOST_SIM_DIST": distances, "DEBUG_MOST_SIM_POS": pos}, index=group.index)
simliarity_features_vd = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(find_most_similiar, word_sim_df)
clean_features = simliarity_features_vd.droplevel([0,1,2])
df = df.join(clean_features)

C:\Users\elito\AppData\Local\Temp\ipykernel_11808\2983514400.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  simliarity_features_vd = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(find_most_similiar, word_sim_df)


In [75]:
df.head()

,index,ListID,Presented Word,Present Position,Hit,Participant ID,Participant Group,POSITION_ZONE,MOST_SIM_VAL,MOST_SIM_DIST,DEBUG_MOST_SIM_POS
0,3231,1,SARI,1,1,0_ENG,eng,1,NaN,NaN,NaN
1,3232,1,KAĞIT,2,1,0_ENG,eng,1,0.381701,1.0,0.0
2,3233,1,YAZI,3,1,0_ENG,eng,1,0.385806,1.0,1.0
3,3234,1,EKRAN,4,0,0_ENG,eng,1,0.252005,1.0,2.0
4,3235,1,PARK,5,0,0_ENG,eng,1,0.314455,3.0,1.0


### Hit - Miss Feature

In [76]:
#redacted due leakage
"""#hit or miss of previous trial 
def prev_hit_miss(group): 
    length = len(group)-1
    hit_miss = []
    hit_list = group["Hit"].values #get rid of pandas indexing
    
    i = 0
    while i <= length: 
        cur = i 
        cur_val = hit_list[i]
        if i > 0:
            hit_miss.append(hit_list[i-1])
        else:
            hit_miss.append(np.nan)
        i+=1
    return pd.DataFrame(data = {"PREV_HIT_MISS": hit_miss}, index=group.index)"""

'#hit or miss of previous trial \ndef prev_hit_miss(group): \n    length = len(group)-1\n    hit_miss = []\n    hit_list = group["Hit"].values #get rid of pandas indexing\n\n    i = 0\n    while i <= length: \n        cur = i \n        cur_val = hit_list[i]\n        if i > 0:\n            hit_miss.append(hit_list[i-1])\n        else:\n            hit_miss.append(np.nan)\n        i+=1\n    return pd.DataFrame(data = {"PREV_HIT_MISS": hit_miss}, index=group.index)'

In [77]:
"""hit_miss_feature = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(prev_hit_miss)
hit_miss_feature = hit_miss_feature.droplevel([0,1,2])
df = df.join(hit_miss_feature)
df.head()"""

'hit_miss_feature = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(prev_hit_miss)\nhit_miss_feature = hit_miss_feature.droplevel([0,1,2])\ndf = df.join(hit_miss_feature)\ndf.head()'

### AVG Similarity With Previous n Words

In [78]:
def avg_sim_previous(group, sim_df, n): 
    length = len(group)
    avg_similarities = []
    words = group["Presented Word"].values 

    i = 0
    while i < length:
        current_word = words[i]
        
        # THE FIX: This determines the start of your look-back window.
        # If i=5 and n=3, start_idx is 2. (It checks j=2, 3, 4)
        # If i=1 and n=3, start_idx is 0. (It checks j=0)
        start_idx = max(0, i - n)
        
        sim_sum = 0
        valid_count = 0
        
        # Loop ONLY from the start_idx up to the current word
        j = start_idx
        while j < i:
            comparison_word = words[j]
            try:
                similarity = sim_df.loc[current_word, comparison_word]
                
                # If the similarity exists, add it to our running total
                if pd.notna(similarity):
                    sim_sum += similarity
                    valid_count += 1
            except KeyError:
                pass
            j += 1
            
        # Calculate the average. If no valid words were found (or i=0), return NaN.
        if valid_count > 0:
            avg_similarities.append(sim_sum / valid_count)
        else:
            avg_similarities.append(np.nan)
            
        i += 1
        
    # Return a DataFrame with a dynamic column name based on 'n'
    return pd.DataFrame(
        data={f"AVG_SIM_PREV_{n}": avg_similarities}, 
        index=group.index
    )
avg_similarity_n = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(avg_sim_previous, word_sim_df, 3)
avg_similarity_n = avg_similarity_n.droplevel([0,1,2])
df = df.join(avg_similarity_n)
df.head()

C:\Users\elito\AppData\Local\Temp\ipykernel_11808\3472631154.py:46: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  avg_similarity_n = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(avg_sim_previous, word_sim_df, 3)


,index,ListID,Presented Word,Present Position,Hit,Participant ID,Participant Group,POSITION_ZONE,MOST_SIM_VAL,MOST_SIM_DIST,DEBUG_MOST_SIM_POS,AVG_SIM_PREV_3
0,3231,1,SARI,1,1,0_ENG,eng,1,NaN,NaN,NaN,NaN
1,3232,1,KAĞIT,2,1,0_ENG,eng,1,0.381701,1.0,0.0,0.381701
2,3233,1,YAZI,3,1,0_ENG,eng,1,0.385806,1.0,1.0,0.325940
3,3234,1,EKRAN,4,0,0_ENG,eng,1,0.252005,1.0,2.0,0.227335
4,3235,1,PARK,5,0,0_ENG,eng,1,0.314455,3.0,1.0,0.272572


### Similarity to Last Word

In [79]:
def sim_last_word(group, sim_df): 
    length = len(group)-1
    similarities = []
    words = group["Presented Word"].values #get rid of pandas indexing
    
    i = 0
    while i <= length:
        current_word = words[i]
        if i>0: 
            comparison_word = words[i-1]
            try: 
                similarity = sim_df.loc[current_word, comparison_word]
                similarities.append(similarity)
            except KeyError: 
                similarities.append(np.nan)
        else: 
            similarities.append(np.nan)
        i += 1
    return pd.DataFrame(data = {"PREV_SIM": similarities}, index = group.index)
prev_word_sim = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(sim_last_word, word_sim_df)
prev_word_sim = prev_word_sim.droplevel([0,1,2])
df = df.join(prev_word_sim)
df.head()

C:\Users\elito\AppData\Local\Temp\ipykernel_11808\1498252722.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  prev_word_sim = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(sim_last_word, word_sim_df)


,index,ListID,Presented Word,Present Position,Hit,Participant ID,Participant Group,POSITION_ZONE,MOST_SIM_VAL,MOST_SIM_DIST,DEBUG_MOST_SIM_POS,AVG_SIM_PREV_3,PREV_SIM
0,3231,1,SARI,1,1,0_ENG,eng,1,NaN,NaN,NaN,NaN,NaN
1,3232,1,KAĞIT,2,1,0_ENG,eng,1,0.381701,1.0,0.0,0.381701,0.381701
2,3233,1,YAZI,3,1,0_ENG,eng,1,0.385806,1.0,1.0,0.325940,0.385806
3,3234,1,EKRAN,4,0,0_ENG,eng,1,0.252005,1.0,2.0,0.227335,0.252005
4,3235,1,PARK,5,0,0_ENG,eng,1,0.314455,3.0,1.0,0.272572,0.222346


### Contrast
How much words semantic content differs from all the rest. 

In [80]:
def contrast(group, vector_mapping, n): 
    length = len(group)
    distances = []
    words = group["Presented Word"].values
    
    i = 0
    while i < length:
        current_word = words[i]
        
        # 1. Safely fetch the target vector first. 
        # If the target word isn't in FastText, we can't calculate a distance.
        try:
            current_word_vector = vector_mapping[current_word]
        except KeyError:
            distances.append(np.nan)
            i += 1
            continue

        # 2. Safe Boundaries (prevents IndexError at the end of the list)
        # Using slice-style boundaries so we don't need +1 in the range later
        neighbors_idx_begin = max(0, i - n)
        neighbors_idx_end = min(length, i + n + 1) 
        
        neighbor_vectors = []
        
        # 3. Iterate through the safe window
        for j in range(neighbors_idx_begin, neighbors_idx_end):
            if j == i:
                continue # Skip the target word itself
                
            word_to_fetch = words[j]
            try:
                # Only add the vector if FastText actually has it
                vector = vector_mapping[word_to_fetch]
                neighbor_vectors.append(vector)
            except KeyError:
                pass
                
        # 4. Calculate Centroid and Distance
        # We must ensure we actually found valid neighbors before taking the mean
        if len(neighbor_vectors) > 0:
            centroid = np.mean(neighbor_vectors, axis=0)
            dist = cosine(current_word_vector, centroid)
            distances.append(dist)
        else:
            distances.append(np.nan)
            
        i += 1
    return pd.DataFrame(
        data={f"AVG_CONTRAST_{n}": distances}, 
        index=group.index
    )
contrast_df = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(contrast, vector_mapping,3)
contrast_df = contrast_df.droplevel([0,1,2])
df = df.join(contrast_df)
df.head()

C:\Users\elito\AppData\Local\Temp\ipykernel_11808\1659942737.py:53: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  contrast_df = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(contrast, vector_mapping,3)


,index,ListID,Presented Word,Present Position,Hit,Participant ID,Participant Group,POSITION_ZONE,MOST_SIM_VAL,MOST_SIM_DIST,DEBUG_MOST_SIM_POS,AVG_SIM_PREV_3,PREV_SIM,AVG_CONTRAST_3
0,3231,1,SARI,1,1,0_ENG,eng,1,NaN,NaN,NaN,NaN,NaN,0.615852
1,3232,1,KAĞIT,2,1,0_ENG,eng,1,0.381701,1.0,0.0,0.381701,0.381701,0.504262
2,3233,1,YAZI,3,1,0_ENG,eng,1,0.385806,1.0,1.0,0.325940,0.385806,0.567887
3,3234,1,EKRAN,4,0,0_ENG,eng,1,0.252005,1.0,2.0,0.227335,0.252005,0.655217
4,3235,1,PARK,5,0,0_ENG,eng,1,0.314455,3.0,1.0,0.272572,0.222346,0.558633


### Fatigue

In [81]:
def fatigue(group): 
    length = len(group)
    fatigue = [i for i in range(1, length+1)]
    return pd.DataFrame({"FATIGUE":fatigue}, index=group.index)

fatigue_df = df.groupby(["Participant ID", "Participant Group"]).apply(fatigue)
fatigue_df = fatigue_df.droplevel([0,1])
df = df.join(fatigue_df)
df.head()

C:\Users\elito\AppData\Local\Temp\ipykernel_11808\3272009529.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fatigue_df = df.groupby(["Participant ID", "Participant Group"]).apply(fatigue)


,index,ListID,Presented Word,Present Position,Hit,Participant ID,Participant Group,POSITION_ZONE,MOST_SIM_VAL,MOST_SIM_DIST,DEBUG_MOST_SIM_POS,AVG_SIM_PREV_3,PREV_SIM,AVG_CONTRAST_3,FATIGUE
0,3231,1,SARI,1,1,0_ENG,eng,1,NaN,NaN,NaN,NaN,NaN,0.615852,1
1,3232,1,KAĞIT,2,1,0_ENG,eng,1,0.381701,1.0,0.0,0.381701,0.381701,0.504262,2
2,3233,1,YAZI,3,1,0_ENG,eng,1,0.385806,1.0,1.0,0.325940,0.385806,0.567887,3
3,3234,1,EKRAN,4,0,0_ENG,eng,1,0.252005,1.0,2.0,0.227335,0.252005,0.655217,4
4,3235,1,PARK,5,0,0_ENG,eng,1,0.314455,3.0,1.0,0.272572,0.222346,0.558633,5


In [83]:
df = df.drop("DEBUG_MOST_SIM_POS", axis = 1)

In [ ]:
df.head()

In [84]:
df.to_csv("features_extracted_2.csv", encoding="utf-8-sig")